In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 16
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
13.877784515136073

Trial 1 =========================================
14.464749305566805

Trial 2 =========================================
14.262714552046699

Trial 3 =========================================
14.090438843773676

Trial 4 =========================================
14.466163399808977

Trial 5 =========================================
16.237429043139286

Trial 6 =========================================
14.285504237988778

Trial 7 =========================================
17.72019647942041



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:677: RuntimeWarning: Optimization failed in `gen_candidates_scipy` with the following warning(s):
[OptimizationWarning('Optimization failed within `scipy.optimize.minimize` with status 2 and message ABNORMAL: .')]
Trying again with a new set of initial conditions.
  return _optimize_acqf_batch(opt_inputs=opt_inputs)


Trial 8 =========================================
16.133224419395503



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 9 =========================================
16.252126494081008

Trial 10 =========================================
13.647840588825755

Trial 11 =========================================
14.61240782724951

Trial 12 =========================================
16.952650600632644

Trial 13 =========================================
18.820524394515907

Trial 14 =========================================
14.454977256473493

Trial 15 =========================================
14.2603509566416

Trial 16 =========================================
16.264815156955077

Trial 17 =========================================
17.635141580928

Trial 18 =========================================
13.39326894296479

Trial 19 =========================================
14.865015229513013

Trial 20 =========================================
14.286114445335635

Trial 21 =========================================
14.357101808697648

Trial 22 =========================================
16.23769734307041



...

Trial 76 =========================================
15.955238420647143

Trial 77 =========================================
15.361191669286118

Trial 78 =========================================
13.849324934276884

Trial 79 =========================================
16.222035588361322

Trial 80 =========================================
13.51385233961432

Trial 81 =========================================
13.891747520615453

Trial 82 =========================================
14.632813212716732

Trial 83 =========================================
17.012981528569448

Trial 84 =========================================
13.8792390354199

Trial 85 =========================================
16.243382386990497

Trial 86 =========================================
15.385533378145285

Trial 87 =========================================
15.335417233023826

Trial 88 =========================================
13.566893032921158

Trial 89 =========================================
14.45436082607205

Trial 90 =

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 92 =========================================
15.987238970836792

Trial 93 =========================================
14.422871519437242



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 94 =========================================
14.225677958468637

Trial 95 =========================================
15.908561330180486

Trial 96 =========================================
13.791481898320331

Trial 97 =========================================
14.290160749380581

Trial 98 =========================================
14.655761351139304

Trial 99 =========================================
14.241922142242242

[13.87778452 14.46474931 14.26271455 14.09043884 14.4661634  16.23742904
 14.28550424 17.72019648 16.13322442 16.25212649 13.64784059 14.61240783
 16.9526506  18.82052439 14.45497726 14.26035096 16.26481516 17.63514158
 13.39326894 14.86501523 14.28611445 14.35710181 16.23769734 16.24079139
 16.21697032 16.19377424 14.85354185 17.57548535 14.15222467 15.98527001
 16.23161658 13.07460072 13.50295284 13.88810147 18.35930793 16.14986276
 18.8222016  15.37286868 14.33743143 14.59277829 17.48638149 15.39797124
 14.36000018 14.89541308 15.36809218 15.92922201 13.97006418 16

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.822201604757687
Avg = 15.333448153385751
Std = 1.3355475866347266


In [6]:
print(y_max_arr.tolist())

[13.877784515136073, 14.464749305566805, 14.262714552046699, 14.090438843773676, 14.466163399808977, 16.237429043139286, 14.285504237988778, 17.72019647942041, 16.133224419395503, 16.252126494081008, 13.647840588825755, 14.61240782724951, 16.952650600632644, 18.820524394515907, 14.454977256473493, 14.2603509566416, 16.264815156955077, 17.635141580928, 13.39326894296479, 14.865015229513013, 14.286114445335635, 14.357101808697648, 16.23769734307041, 16.240791390202805, 16.21697031916443, 16.19377424070045, 14.853541846314974, 17.575485353295445, 14.152224672508293, 15.985270010412261, 16.231616580165678, 13.074600719832395, 13.502952841334515, 13.88810146983019, 18.359307931992426, 16.149862758370872, 18.822201604757687, 15.372868682578279, 14.33743143376383, 14.592778294659421, 17.48638149384863, 15.397971240295586, 14.360000183278624, 14.895413080548074, 15.368092177039816, 15.929222010399386, 13.970064177666321, 16.23576711957996, 14.376280760784415, 17.47254684198196, 14.263179902641

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-16.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)